# Quantización + fine-tuning MLP27K Montmeló para FINN v3

Este notebook hace el flujo siguiente:

1. Carga el checkpoint bueno `mlp_27k.ckpt`.
2. Comprueba el modelo float contra el CSV.
3. Crea la mejor cuantización directa encontrada: `Q_NoOut_w4_a8_act1`.
4. La evalúa contra el CSV.
5. Hace **fine-tuning cuantizado** para intentar recuperar precisión.
6. Exporta el modelo cuantizado ajustado a ONNX FINN.
7. Opcionalmente crea una versión con preprocesado `/255` para entrada `UINT8`.
8. Intenta `build_dataflow` y deja celdas de debug si FINN falla.

Antes de ejecutar, asegúrate de que en la carpeta del notebook está:

```text
mlp_27k.ckpt
```


## Versión first500

Este notebook está preparado para comprobar las **500 primeras imágenes** en las celdas de evaluación.

Cambios principales:

- `rows_eval = load_rows(500)` para evaluar solo las 500 primeras muestras.
- `print_first=500` en las evaluaciones de PyTorch y ONNX.
- En la comprobación final de `model.onnx` también se imprimen las primeras 500.

## 0. Imports

In [1]:
from pathlib import Path
import os
import csv
import math
import random
import shutil
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import brevitas.nn as qnn
import brevitas.onnx as bo
from brevitas.quant import Int8Bias

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.core.datatype import DataType
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import (
    GiveUniqueNodeNames,
    GiveReadableTensorNames,
    RemoveStaticGraphInputs,
    RemoveUnusedTensors,
)

print("Torch:", torch.__version__)
print("Working dir:", Path.cwd())


Torch: 1.7.1
Working dir: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example


## 1. Rutas y dataset

In [2]:
CKPT_PATH = Path("mlp_27k.ckpt")

candidate_dirs = [
    Path("output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic"),
    Path("/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic"),
    Path("/tmp/Orig/DATA_12_16/counter_montmelo_holonomic"),
    Path("/tmp/Orig/Orig/DATA_12_16/counter_montmelo_holonomic"),
]

DATASET_DIR = None
for d in candidate_dirs:
    if (d / "data.csv").exists():
        DATASET_DIR = d
        break

if not CKPT_PATH.exists():
    raise FileNotFoundError("No encuentro mlp_27k.ckpt en la carpeta del notebook")

if DATASET_DIR is None:
    raise FileNotFoundError("No encuentro el data.csv del dataset")

CSV_PATH = DATASET_DIR / "data.csv"
print("Checkpoint:", CKPT_PATH.resolve())
print("Dataset:", DATASET_DIR)
print("CSV:", CSV_PATH)

VMIN = -0.6
VMAX = 13.0
WMIN = -math.pi
WMAX = math.pi


def denorm_v_01(vnorm):
    return (VMAX - VMIN) * vnorm + VMIN


def denorm_w_01(wnorm):
    return (WMAX - WMIN) * wnorm + WMIN


def norm_v_01(v):
    return (v - VMIN) / (VMAX - VMIN)


def norm_w_01(w):
    return (w - WMIN) / (WMAX - WMIN)


def image_to_flat_01(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)  # RRR GGG BBB
    arr = arr / 255.0
    return torch.tensor(arr, dtype=torch.float32).unsqueeze(0)


def image_to_flat_uint8_np(img_path):
    img = Image.open(img_path).convert("RGB")
    img = img.resize((16, 12))
    arr = np.asarray(img).astype(np.float32)
    arr = np.transpose(arr, (2, 0, 1)).reshape(-1)  # RRR GGG BBB
    return arr.reshape(1, 576).astype(np.float32)


def load_rows(max_rows=None):
    rows = []
    with open(CSV_PATH) as f:
        reader = csv.DictReader(f)
        for row in reader:
            img_path = DATASET_DIR / row["image_name"]
            if not img_path.exists():
                continue
            rows.append({
                "image_name": row["image_name"],
                "img_path": img_path,
                "v": float(row["v"]),
                "w": float(row["w"]),
            })
            if max_rows is not None and len(rows) >= max_rows:
                break
    return rows

rows_eval = load_rows(500)  # primeras 500 imágenes
rows_all = load_rows(None)
print("Frames eval:", len(rows_eval))
print("Frames total:", len(rows_all))


Checkpoint: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/mlp_27k.ckpt
Dataset: output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic
CSV: output_final_miguel/Dataset_JdeRobot/DATA_12_16/counter_montmelo_holonomic/data.csv
Frames eval: 500
Frames total: 6539


## 2. Modelo float de referencia

In [3]:
class FloatMLP27K(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(576, 32)
        self.fx = nn.Linear(32, 256)
        self.predictor = nn.Linear(256, 2)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fx(x))
        x = self.predictor(x)
        return x


def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ["state_dict", "model_state_dict", "net", "model"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
        return obj
    return obj.state_dict()


def clean_state_dict(sd):
    out = {}
    for k, v in sd.items():
        if not torch.is_tensor(v):
            continue
        nk = k
        changed = True
        while changed:
            changed = False
            for pref in ["model.", "base_model.", "module.", "net."]:
                if nk.startswith(pref):
                    nk = nk[len(pref):]
                    changed = True
        nk = nk.replace("predictor.0.", "predictor.")
        out[nk] = v
    return out


obj = torch.load(str(CKPT_PATH), map_location="cpu")
sd = clean_state_dict(extract_state_dict(obj))

print("Tensores encontrados en checkpoint:")
for k, v in sd.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape))

float_model = FloatMLP27K()
float_model.load_state_dict(sd, strict=True)
float_model.eval()
print("Modelo float cargado OK")


Tensores encontrados en checkpoint:
fc1.weight (32, 576)
fc1.bias (32,)
fx.weight (256, 32)
fx.bias (256,)
predictor.weight (2, 256)
predictor.bias (2,)
Modelo float cargado OK


## 3. Función de evaluación contra CSV

In [4]:
def eval_pytorch_model(model, rows, name="model", print_first=500):
    model.eval()
    err_v, err_w = [], []
    pred_v, pred_w = [], []

    with torch.no_grad():
        for i, r in enumerate(rows):
            x = image_to_flat_01(r["img_path"])
            y = model(x)
            if hasattr(y, "value"):
                y = y.value
            y = y.view(-1).detach().cpu().numpy()

            vnorm = float(y[0])
            wnorm = float(y[1])
            v_pred = denorm_v_01(vnorm)
            w_pred = denorm_w_01(wnorm)

            v_csv = r["v"]
            w_csv = r["w"]

            err_v.append(v_pred - v_csv)
            err_w.append(w_pred - w_csv)
            pred_v.append(v_pred)
            pred_w.append(w_pred)

            if i < print_first:
                raw_v_csv = round(norm_v_01(v_csv) * 255)
                raw_w_csv = round(norm_w_01(w_csv) * 255)
                raw_v_pred = round(vnorm * 255)
                raw_w_pred = round(wnorm * 255)
                print(
                    f"{r['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} "
                    f"raw_csv={raw_v_csv},{raw_w_csv} | "
                    f"{name} v={v_pred:.4f} w={w_pred:.4f} raw={raw_v_pred},{raw_w_pred}"
                )

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    res = {
        "name": name,
        "frames": len(rows),
        "mae_v": float(np.mean(np.abs(err_v))),
        "mae_w": float(np.mean(np.abs(err_w))),
        "std_v": float(np.std(pred_v)),
        "std_w": float(np.std(pred_w)),
        "v_min": float(np.min(pred_v)),
        "v_max": float(np.max(pred_v)),
        "w_min": float(np.min(pred_w)),
        "w_max": float(np.max(pred_w)),
    }

    print()
    print(name)
    print("Frames:", res["frames"])
    print("MAE_v =", res["mae_v"])
    print("MAE_w =", res["mae_w"])
    print("v pred min/max/std =", res["v_min"], res["v_max"], res["std_v"])
    print("w pred min/max/std =", res["w_min"], res["w_max"], res["std_w"])

    return res


## 4. Evaluar modelo float

In [5]:
float_res = eval_pytorch_model(float_model, rows_eval, name="FloatMLP27K", print_first=500)
float_res


1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | FloatMLP27K v=4.8432 w=-0.1681 raw=102,121
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.1680 raw=102,121
11.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | FloatMLP27K v=4.8432 w=-0.168

{'name': 'FloatMLP27K',
 'frames': 500,
 'mae_v': 0.7007640474319455,
 'mae_w': 0.021042273417337328,
 'std_v': 0.0005881802632755762,
 'std_w': 0.06454991196686344,
 'v_min': 4.839579367637635,
 'v_max': 4.844488501548767,
 'w_min': -0.16820402689858982,
 'w_max': 0.15616156615033372}

## 5. Modelo cuantizado elegido: pesos 4 bits, activaciones 8 bits

In [6]:
class QuantMLP27K_W4A8_NoQuantOut(nn.Module):
    def __init__(self):
        super().__init__()

        self.quant_in = qnn.QuantIdentity(
            bit_width=8,
            return_quant_tensor=True,
            min_val=0.0,
            max_val=1.0
        )

        self.fc1 = qnn.QuantLinear(
            576, 32,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )
        self.relu1 = qnn.QuantReLU(
            bit_width=8,
            return_quant_tensor=True
        )

        self.fx = qnn.QuantLinear(
            32, 256,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )
        self.relu2 = qnn.QuantReLU(
            bit_width=8,
            return_quant_tensor=True
        )

        self.predictor = qnn.QuantLinear(
            256, 2,
            bias=True,
            weight_bit_width=4,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.quant_in(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fx(x)
        x = self.relu2(x)
        x = self.predictor(x)
        if hasattr(x, "value"):
            x = x.value
        return x


def load_float_weights_into_quant(qmodel, sd):
    qsd = qmodel.state_dict()
    loaded = []
    for k, v in sd.items():
        if k in qsd and tuple(qsd[k].shape) == tuple(v.shape):
            qsd[k] = v
            loaded.append(k)
    qmodel.load_state_dict(qsd, strict=False)
    return loaded


qmodel = QuantMLP27K_W4A8_NoQuantOut()
loaded = load_float_weights_into_quant(qmodel, sd)
qmodel.eval()
print("Pesos cargados en cuantizado:")
for k in loaded:
    print(" ", k)
print("Total:", len(loaded))


Pesos cargados en cuantizado:
  fc1.weight
  fc1.bias
  fx.weight
  fx.bias
  predictor.weight
  predictor.bias
Total: 6


## 6. Evaluar cuantizado antes de fine-tuning

In [7]:
q_before_res = eval_pytorch_model(qmodel, rows_eval, name="Q_W4A8_before_finetune", print_first=500)
q_before_res


1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune v=5.0482 w=-0.3801 raw=106,112
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_before_finetune 

84.png | CSV v=5.0000 w=0.1647 raw_csv=105,134 | Q_W4A8_before_finetune v=4.6598 w=-0.2155 raw=99,119
85.png | CSV v=5.0000 w=0.1650 raw_csv=105,134 | Q_W4A8_before_finetune v=4.6598 w=-0.2155 raw=99,119
86.png | CSV v=5.0000 w=0.1597 raw_csv=105,134 | Q_W4A8_before_finetune v=4.6735 w=-0.2049 raw=99,119
87.png | CSV v=5.0000 w=0.1600 raw_csv=105,134 | Q_W4A8_before_finetune v=4.6735 w=-0.2049 raw=99,119
88.png | CSV v=5.0000 w=0.1547 raw_csv=105,134 | Q_W4A8_before_finetune v=4.6780 w=-0.2155 raw=99,119
89.png | CSV v=5.0000 w=0.1550 raw_csv=105,134 | Q_W4A8_before_finetune v=4.6780 w=-0.2155 raw=99,119
90.png | CSV v=5.0000 w=0.1550 raw_csv=105,134 | Q_W4A8_before_finetune v=4.6780 w=-0.2281 raw=99,118
91.png | CSV v=5.0000 w=0.1550 raw_csv=105,134 | Q_W4A8_before_finetune v=4.6780 w=-0.2281 raw=99,118
92.png | CSV v=5.0000 w=0.1497 raw_csv=105,134 | Q_W4A8_before_finetune v=4.7009 w=-0.2028 raw=99,119
93.png | CSV v=5.0000 w=0.1500 raw_csv=105,134 | Q_W4A8_before_finetune v=4.7009 w

166.png | CSV v=5.0000 w=0.1100 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7557 w=-0.2302 raw=100,118
167.png | CSV v=5.0000 w=0.1100 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7557 w=-0.2302 raw=100,118
168.png | CSV v=5.0000 w=0.1100 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7603 w=-0.2302 raw=101,118
169.png | CSV v=5.0000 w=0.1100 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7603 w=-0.2302 raw=101,118
170.png | CSV v=5.0000 w=0.1047 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7603 w=-0.2366 raw=101,118
171.png | CSV v=5.0000 w=0.1050 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7603 w=-0.2366 raw=101,118
172.png | CSV v=5.0000 w=0.1050 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7557 w=-0.2408 raw=100,118
173.png | CSV v=5.0000 w=0.1050 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7557 w=-0.2408 raw=100,118
174.png | CSV v=5.0000 w=0.1050 raw_csv=105,132 | Q_W4A8_before_finetune v=4.7557 w=-0.2408 raw=100,118
175.png | CSV v=5.0000 w=0.1050 raw_csv=105,132 | Q_W4A8_before_

257.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.8197 w=-0.2387 raw=102,118
258.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.7969 w=-0.2429 raw=101,118
259.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.7969 w=-0.2429 raw=101,118
260.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.7969 w=-0.2429 raw=101,118
261.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.7969 w=-0.2429 raw=101,118
262.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.7969 w=-0.2471 raw=101,117
263.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.8288 w=-0.2323 raw=102,118
264.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.8288 w=-0.2323 raw=102,118
265.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_finetune v=4.8288 w=-0.2323 raw=102,118
266.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_before_

347.png | CSV v=5.0000 w=0.0450 raw_csv=105,129 | Q_W4A8_before_finetune v=4.8745 w=-0.2450 raw=103,118
348.png | CSV v=5.0000 w=0.0450 raw_csv=105,129 | Q_W4A8_before_finetune v=4.8745 w=-0.2450 raw=103,118
349.png | CSV v=5.0000 w=0.0450 raw_csv=105,129 | Q_W4A8_before_finetune v=4.8745 w=-0.2450 raw=103,118
350.png | CSV v=5.0000 w=0.0450 raw_csv=105,129 | Q_W4A8_before_finetune v=4.8745 w=-0.2450 raw=103,118
351.png | CSV v=5.0000 w=0.0450 raw_csv=105,129 | Q_W4A8_before_finetune v=4.9020 w=-0.2408 raw=103,118
352.png | CSV v=5.0000 w=0.0397 raw_csv=105,129 | Q_W4A8_before_finetune v=4.9111 w=-0.2218 raw=103,118
353.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_before_finetune v=4.9111 w=-0.2218 raw=103,118
354.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_before_finetune v=4.9020 w=-0.2366 raw=103,118
355.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_before_finetune v=4.9020 w=-0.2366 raw=103,118
356.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_before_

427.png | CSV v=5.0000 w=-0.0050 raw_csv=105,127 | Q_W4A8_before_finetune v=5.0025 w=-0.2640 raw=105,117
428.png | CSV v=5.0000 w=-0.0050 raw_csv=105,127 | Q_W4A8_before_finetune v=5.0025 w=-0.2640 raw=105,117
429.png | CSV v=5.0000 w=-0.0050 raw_csv=105,127 | Q_W4A8_before_finetune v=5.0162 w=-0.2577 raw=105,117
430.png | CSV v=5.0000 w=-0.0050 raw_csv=105,127 | Q_W4A8_before_finetune v=5.0345 w=-0.2682 raw=106,117
431.png | CSV v=5.0000 w=-0.0050 raw_csv=105,127 | Q_W4A8_before_finetune v=5.0345 w=-0.2682 raw=106,117
432.png | CSV v=5.0000 w=-0.0050 raw_csv=105,127 | Q_W4A8_before_finetune v=5.0345 w=-0.2577 raw=106,117
433.png | CSV v=5.0000 w=-0.0050 raw_csv=105,127 | Q_W4A8_before_finetune v=5.0345 w=-0.2577 raw=106,117
434.png | CSV v=13.0000 w=-0.0057 raw_csv=255,127 | Q_W4A8_before_finetune v=5.0391 w=-0.2725 raw=106,116
435.png | CSV v=13.0000 w=-0.0054 raw_csv=255,127 | Q_W4A8_before_finetune v=5.0391 w=-0.2725 raw=106,116
436.png | CSV v=13.0000 w=-0.0054 raw_csv=255,127 | Q

{'name': 'Q_W4A8_before_finetune',
 'frames': 500,
 'mae_v': 0.6931549139022827,
 'mae_w': 0.3038793627654314,
 'std_v': 0.12684538392727712,
 'std_w': 0.036869655836021635,
 'v_min': 4.632344770431518,
 'v_max': 5.112167215347291,
 'w_min': -0.38857238875397027,
 'w_max': -0.19856325189359847}

## 7. Dataset y DataLoaders para fine-tuning

In [8]:
class MontmeloDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        x = image_to_flat_01(r["img_path"]).view(-1)
        y = torch.tensor([norm_v_01(r["v"]), norm_w_01(r["w"])], dtype=torch.float32)
        return x, y


random.seed(0)
rows_train_val = rows_all.copy()
random.shuffle(rows_train_val)

split = int(0.8 * len(rows_train_val))
train_rows = rows_train_val[:split]
val_rows = rows_train_val[split:]

train_ds = MontmeloDataset(train_rows)
val_ds = MontmeloDataset(val_rows)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))


Train samples: 5231
Val samples: 1308


## 8. Fine-tuning cuantizado

In [10]:
device = torch.device("cpu")
qmodel = qmodel.to(device)

loss_fn = nn.L1Loss()
optimizer = torch.optim.Adam(qmodel.parameters(), lr=1e-4, weight_decay=1e-6)

N_EPOCHS = 50
best_val_loss = 1e9
best_state = None

for epoch in range(1, N_EPOCHS + 1):
    qmodel.train()
    train_losses = []

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = qmodel(x)
        if hasattr(pred, "value"):
            pred = pred.value

        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    qmodel.eval()
    val_losses = []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)
            pred = qmodel(x)
            if hasattr(pred, "value"):
                pred = pred.value
            loss = loss_fn(pred, y)
            val_losses.append(float(loss.item()))

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in qmodel.state_dict().items()}

    print(f"Epoch {epoch:03d}/{N_EPOCHS} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

print("Mejor val_loss:", best_val_loss)
qmodel.load_state_dict(best_state)
qmodel.eval()


Epoch 001/50 | train_loss=0.027347 | val_loss=0.025286
Epoch 002/50 | train_loss=0.027889 | val_loss=0.026858
Epoch 003/50 | train_loss=0.029230 | val_loss=0.026875
Epoch 004/50 | train_loss=0.027495 | val_loss=0.031003
Epoch 005/50 | train_loss=0.027928 | val_loss=0.027809
Epoch 006/50 | train_loss=0.027907 | val_loss=0.031797
Epoch 007/50 | train_loss=0.028400 | val_loss=0.026421
Epoch 008/50 | train_loss=0.027076 | val_loss=0.027753
Epoch 009/50 | train_loss=0.027673 | val_loss=0.027526
Epoch 010/50 | train_loss=0.027312 | val_loss=0.026745
Epoch 011/50 | train_loss=0.026573 | val_loss=0.026622
Epoch 012/50 | train_loss=0.027679 | val_loss=0.025807
Epoch 013/50 | train_loss=0.026814 | val_loss=0.026203
Epoch 014/50 | train_loss=0.026635 | val_loss=0.026954
Epoch 015/50 | train_loss=0.028291 | val_loss=0.035128
Epoch 016/50 | train_loss=0.027755 | val_loss=0.032520
Epoch 017/50 | train_loss=0.027230 | val_loss=0.026832
Epoch 018/50 | train_loss=0.026167 | val_loss=0.027248
Epoch 019/

QuantMLP27K_W4A8_NoQuantOut(
  (quant_in): QuantIdentity(
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (act_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (fused_activation_quant_proxy): FusedActivationQuantProxy(
        (activation_impl): Identity()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClamp()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): ParameterFromRuntimeStatsScaling(
            (stats_input_view_shape_impl): OverTensorView()
            (stats): _Stats(
              (stats_impl): AbsPercentile()
            )
            (restrict_clamp_scaling): _RestrictClampValue(
              (clamp_min_ste): Identity()
              (restrict_value_impl): FloatRestrictValue()
            

## 9. Evaluar cuantizado después del fine-tuning

In [12]:
q_after_res = eval_pytorch_model(qmodel, rows_eval, name="Q_W4A8_finetuned", print_first=500)
q_after_res


1.png | CSV v=5.0000 w=-0.2254 raw_csv=105,118 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
2.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
3.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
4.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
5.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
6.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
7.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
8.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
9.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
10.png | CSV v=5.0000 w=-0.2131 raw_csv=105,119 | Q_W4A8_finetuned v=4.9801 w=-0.2320 raw=105,118
11.png | CSV v=5.0000 w=-0.21

120.png | CSV v=5.0000 w=0.1250 raw_csv=105,133 | Q_W4A8_finetuned v=5.0392 w=0.0722 raw=106,130
121.png | CSV v=5.0000 w=0.1250 raw_csv=105,133 | Q_W4A8_finetuned v=5.0392 w=0.0722 raw=106,130
122.png | CSV v=5.0000 w=0.1250 raw_csv=105,133 | Q_W4A8_finetuned v=5.0392 w=0.0722 raw=106,130
123.png | CSV v=5.0000 w=0.1250 raw_csv=105,133 | Q_W4A8_finetuned v=5.0392 w=0.0722 raw=106,130
124.png | CSV v=5.0000 w=0.1250 raw_csv=105,133 | Q_W4A8_finetuned v=5.0392 w=0.0722 raw=106,130
125.png | CSV v=5.0000 w=0.1250 raw_csv=105,133 | Q_W4A8_finetuned v=5.0392 w=0.0722 raw=106,130
126.png | CSV v=5.0000 w=0.1250 raw_csv=105,133 | Q_W4A8_finetuned v=5.0433 w=0.0713 raw=106,130
127.png | CSV v=5.0000 w=0.1250 raw_csv=105,133 | Q_W4A8_finetuned v=5.0433 w=0.0713 raw=106,130
128.png | CSV v=5.0000 w=0.1197 raw_csv=105,132 | Q_W4A8_finetuned v=5.0392 w=0.0722 raw=106,130
129.png | CSV v=5.0000 w=0.1200 raw_csv=105,132 | Q_W4A8_finetuned v=5.0392 w=0.0722 raw=106,130
130.png | CSV v=5.0000 w=0.120

252.png | CSV v=5.0000 w=0.0900 raw_csv=105,131 | Q_W4A8_finetuned v=5.0474 w=0.0515 raw=106,130
253.png | CSV v=5.0000 w=0.0847 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
254.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
255.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
256.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
257.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
258.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
259.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
260.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
261.png | CSV v=5.0000 w=0.0850 raw_csv=105,131 | Q_W4A8_finetuned v=5.0535 w=0.0553 raw=106,130
262.png | CSV v=5.0000 w=0.085

353.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_finetuned v=5.0392 w=0.0270 raw=106,129
354.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_finetuned v=5.0372 w=0.0270 raw=106,129
355.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_finetuned v=5.0372 w=0.0270 raw=106,129
356.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_finetuned v=5.0372 w=0.0261 raw=106,129
357.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_finetuned v=5.0372 w=0.0261 raw=106,129
358.png | CSV v=5.0000 w=0.0400 raw_csv=105,129 | Q_W4A8_finetuned v=5.0372 w=0.0261 raw=106,129
359.png | CSV v=5.0000 w=0.0347 raw_csv=105,129 | Q_W4A8_finetuned v=5.0311 w=0.0298 raw=106,129
360.png | CSV v=5.0000 w=0.0350 raw_csv=105,129 | Q_W4A8_finetuned v=5.0392 w=0.0261 raw=106,129
361.png | CSV v=5.0000 w=0.0350 raw_csv=105,129 | Q_W4A8_finetuned v=5.0392 w=0.0261 raw=106,129
362.png | CSV v=5.0000 w=0.0350 raw_csv=105,129 | Q_W4A8_finetuned v=5.0392 w=0.0232 raw=106,128
363.png | CSV v=5.0000 w=0.035

479.png | CSV v=5.0000 w=-0.0500 raw_csv=105,125 | Q_W4A8_finetuned v=5.0250 w=-0.0644 raw=105,125
480.png | CSV v=5.0000 w=-0.0500 raw_csv=105,125 | Q_W4A8_finetuned v=5.0250 w=-0.0644 raw=105,125
481.png | CSV v=5.0000 w=-0.0500 raw_csv=105,125 | Q_W4A8_finetuned v=5.0311 w=-0.0578 raw=106,125
482.png | CSV v=5.0000 w=-0.0500 raw_csv=105,125 | Q_W4A8_finetuned v=5.0311 w=-0.0578 raw=106,125
483.png | CSV v=13.0000 w=-0.0270 raw_csv=255,126 | Q_W4A8_finetuned v=5.0311 w=-0.0578 raw=106,125
484.png | CSV v=13.0000 w=-0.0270 raw_csv=255,126 | Q_W4A8_finetuned v=5.0311 w=-0.0578 raw=106,125
485.png | CSV v=5.0000 w=-0.0447 raw_csv=105,126 | Q_W4A8_finetuned v=5.0311 w=-0.0578 raw=106,125
486.png | CSV v=5.0000 w=-0.0450 raw_csv=105,126 | Q_W4A8_finetuned v=5.0311 w=-0.0578 raw=106,125
487.png | CSV v=5.0000 w=-0.0450 raw_csv=105,126 | Q_W4A8_finetuned v=5.0352 w=-0.0606 raw=106,125
488.png | CSV v=5.0000 w=-0.0450 raw_csv=105,126 | Q_W4A8_finetuned v=5.0352 w=-0.0606 raw=106,125
489.png 

{'name': 'Q_W4A8_finetuned',
 'frames': 500,
 'mae_v': 0.5782901945114137,
 'mae_w': 0.028232503033692007,
 'std_v': 0.015858705448121627,
 'std_w': 0.0763342151520916,
 'v_min': 4.9801215648651125,
 'v_max': 5.061672687530518,
 'w_min': -0.2348556044843293,
 'w_max': 0.20407487182897777}

## 10. Guardar modelo fine-tuned

In [13]:
FT_PTH = Path("q_mlp27k_w4a8_finetuned.pth")
torch.save(qmodel.state_dict(), FT_PTH)
print("Guardado:", FT_PTH)
!ls -lh q_mlp27k_w4a8_finetuned.pth


Guardado: q_mlp27k_w4a8_finetuned.pth
-rw-r--r-- 1 1004 1005 127K Jul  2 16:08 q_mlp27k_w4a8_finetuned.pth


## 11. Exportar a ONNX FINN

In [14]:
FT_ONNX = Path("mlp27k_w4a8_finetuned.onnx")

bo.export_finn_onnx(
    qmodel,
    input_shape=(1, 576),
    export_path=str(FT_ONNX)
)

print("Exportado:", FT_ONNX)
!ls -lh mlp27k_w4a8_finetuned.onnx


Exportado: mlp27k_w4a8_finetuned.onnx
-rw-r--r-- 1 1004 1005 970K Jul  2 16:08 mlp27k_w4a8_finetuned.onnx


## 12. Tidy-up del ONNX cuantizado

In [15]:
MODEL_FOR_FINN_01 = Path("model_for_finn_w4a8_finetuned_01.onnx")

model = ModelWrapper(str(FT_ONNX))
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model = model.transform(RemoveUnusedTensors())
model.save(str(MODEL_FOR_FINN_01))

print("Guardado:", MODEL_FOR_FINN_01)
print("Input:", model.graph.input[0].name, model.get_tensor_shape(model.graph.input[0].name), model.get_tensor_datatype(model.graph.input[0].name))
print("Output:", model.graph.output[0].name, model.get_tensor_shape(model.graph.output[0].name))


Guardado: model_for_finn_w4a8_finetuned_01.onnx
Input: global_in [1, 576] FLOAT32
Output: global_out [1, 2]


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/core/modelwrapper.py:93: UserWarning: Some old-style domain attributes were automatically converted to new-style,
                i.e. domain=finn to domain=qonnx.custom_op.<general|fpgadataflow|...>
  warnings.warn(


## 13. Evaluar ONNX exportado contra CSV

In [17]:
def eval_onnx_01(onnx_path, rows, name="ONNX", print_first=500):
    model = ModelWrapper(str(onnx_path))
    model = model.transform(InferShapes())
    input_name = model.graph.input[0].name
    output_name = model.graph.output[0].name

    err_v, err_w = [], []
    pred_v, pred_w = [], []

    for i, r in enumerate(rows):
        x = image_to_flat_01(r["img_path"]).numpy().astype(np.float32)
        y_dict = execute_onnx(model, {input_name: x})
        y = y_dict[output_name].reshape(-1)

        vnorm = float(y[0])
        wnorm = float(y[1])
        v_pred = denorm_v_01(vnorm)
        w_pred = denorm_w_01(wnorm)
        v_csv = r["v"]
        w_csv = r["w"]

        err_v.append(v_pred - v_csv)
        err_w.append(w_pred - w_csv)
        pred_v.append(v_pred)
        pred_w.append(w_pred)

        if i < print_first:
            print(
                f"{r['image_name']} | CSV v={v_csv:.4f} w={w_csv:.4f} | "
                f"{name} v={v_pred:.4f} w={w_pred:.4f}"
            )

    err_v = np.array(err_v)
    err_w = np.array(err_w)
    pred_v = np.array(pred_v)
    pred_w = np.array(pred_w)

    res = {
        "name": name,
        "frames": len(rows),
        "mae_v": float(np.mean(np.abs(err_v))),
        "mae_w": float(np.mean(np.abs(err_w))),
        "std_v": float(np.std(pred_v)),
        "std_w": float(np.std(pred_w)),
    }
    print(res)
    return res

onnx_res = eval_onnx_01(MODEL_FOR_FINN_01, rows_eval, name="ONNX_W4A8_finetuned", print_first=500)
onnx_res


1.png | CSV v=5.0000 w=-0.2254 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
2.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
3.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
4.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
5.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
6.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
7.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
8.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
9.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
10.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
11.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9883 w=-0.2349
12.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
13.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.9801 w=-0.2320
14.png | CSV v=5.0000 w=-0.2131 | ONNX_W4A8_finetuned v=4.98

116.png | CSV v=5.0000 w=0.1300 | ONNX_W4A8_finetuned v=5.0576 w=0.0835
117.png | CSV v=5.0000 w=0.1300 | ONNX_W4A8_finetuned v=5.0576 w=0.0835
118.png | CSV v=5.0000 w=0.1247 | ONNX_W4A8_finetuned v=5.0392 w=0.0779
119.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0392 w=0.0779
120.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0392 w=0.0722
121.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0392 w=0.0722
122.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0392 w=0.0722
123.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0392 w=0.0722
124.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0392 w=0.0722
125.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0392 w=0.0722
126.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0433 w=0.0713
127.png | CSV v=5.0000 w=0.1250 | ONNX_W4A8_finetuned v=5.0433 w=0.0713
128.png | CSV v=5.0000 w=0.1197 | ONNX_W4A8_finetuned v=5.0392 w=0.0722
129.png | CSV v=5.0000 w=0.1200 | ONNX_W4A8_finetuned v=5.0392 w

230.png | CSV v=5.0000 w=0.0950 | ONNX_W4A8_finetuned v=5.0494 w=0.0534
231.png | CSV v=5.0000 w=0.0950 | ONNX_W4A8_finetuned v=5.0494 w=0.0534
232.png | CSV v=5.0000 w=0.0950 | ONNX_W4A8_finetuned v=5.0454 w=0.0477
233.png | CSV v=5.0000 w=0.0950 | ONNX_W4A8_finetuned v=5.0454 w=0.0477
234.png | CSV v=5.0000 w=0.0897 | ONNX_W4A8_finetuned v=5.0494 w=0.0534
235.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0494 w=0.0534
236.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0454 w=0.0505
237.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0454 w=0.0505
238.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0494 w=0.0534
239.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0494 w=0.0534
240.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0454 w=0.0505
241.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0454 w=0.0505
242.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0454 w=0.0505
243.png | CSV v=5.0000 w=0.0900 | ONNX_W4A8_finetuned v=5.0454 w

KeyboardInterrupt: 

## 14. Crear versión con preprocesado /255 para entrada UINT8

In [18]:
# ============================================================
# Crear modelo UINT8 con preprocesado /255
# Versión con prints para saber dónde se queda
# ============================================================

from pathlib import Path
import shutil
import time

print("Inicio celda UINT8...")

try:
    from finn.util.pytorch import ToTensor
    from qonnx.transformation.merge_onnx_models import MergeONNXModels

    MODEL_FOR_FINN_UINT8 = Path("model_for_finn_w4a8_finetuned_uint8.onnx")
    PREPROC_ONNX = Path("preproc_div255_w4a8.onnx")

    print("1) Cargando modelo 0-1:", MODEL_FOR_FINN_01)
    model = ModelWrapper(str(MODEL_FOR_FINN_01))

    print("2) InferShapes modelo 0-1...")
    model = model.transform(InferShapes())

    global_inp_name = model.graph.input[0].name
    ishape = model.get_tensor_shape(global_inp_name)

    print("Input modelo 0-1:", global_inp_name)
    print("Shape modelo 0-1:", ishape)
    print("Datatype entrada modelo 0-1:", model.get_tensor_datatype(global_inp_name))

    print("3) Exportando preprocesado ToTensor /255...")
    t0 = time.time()

    totensor_pyt = ToTensor()

    bo.export_finn_onnx(
        totensor_pyt,
        ishape,
        str(PREPROC_ONNX)
    )

    print("Preproc exportado en", time.time() - t0, "s")
    print("Existe preproc:", PREPROC_ONNX.exists(), PREPROC_ONNX)

    print("4) Cargando preproc...")
    pre_model = ModelWrapper(str(PREPROC_ONNX))

    print("5) MergeONNXModels...")
    t0 = time.time()

    merged = model.transform(MergeONNXModels(pre_model))

    print("Merge terminado en", time.time() - t0, "s")

    print("6) Ajustando datatype entrada a UINT8...")
    global_inp_name = merged.graph.input[0].name
    merged.set_tensor_datatype(global_inp_name, DataType["UINT8"])

    print("7) Limpieza básica...")
    merged = merged.transform(InferShapes())
    print("   InferShapes OK")

    merged = merged.transform(FoldConstants())
    print("   FoldConstants OK")

    merged = merged.transform(GiveUniqueNodeNames())
    print("   GiveUniqueNodeNames OK")

    merged = merged.transform(GiveReadableTensorNames())
    print("   GiveReadableTensorNames OK")

    merged = merged.transform(InferDataTypes())
    print("   InferDataTypes OK")

    merged = merged.transform(RemoveStaticGraphInputs())
    print("   RemoveStaticGraphInputs OK")

    merged = merged.transform(RemoveUnusedTensors())
    print("   RemoveUnusedTensors OK")

    print("8) Guardando modelo UINT8...")
    merged.save(str(MODEL_FOR_FINN_UINT8))

    print("9) Copiando como model.onnx...")
    shutil.copyfile(MODEL_FOR_FINN_UINT8, "model.onnx")

    print("Modelo UINT8 para FINN guardado en:", MODEL_FOR_FINN_UINT8)
    print("Copiado como model.onnx")
    print(
        "Input:",
        merged.graph.input[0].name,
        merged.get_tensor_shape(merged.graph.input[0].name),
        merged.get_tensor_datatype(merged.graph.input[0].name)
    )

except Exception as e:
    print("No se pudo crear la versión UINT8 con preprocesado.")
    print("Error:", repr(e))
    print("Copio el modelo 0-1 como model.onnx")
    shutil.copyfile(MODEL_FOR_FINN_01, "model.onnx")
    print("Copiado:", MODEL_FOR_FINN_01, "-> model.onnx")


Inicio celda UINT8...
1) Cargando modelo 0-1: model_for_finn_w4a8_finetuned_01.onnx
2) InferShapes modelo 0-1...
Input modelo 0-1: global_in
Shape modelo 0-1: [1, 576]
Datatype entrada modelo 0-1: FLOAT32
3) Exportando preprocesado ToTensor /255...
Preproc exportado en 0.002330780029296875 s
Existe preproc: True preproc_div255_w4a8.onnx
4) Cargando preproc...
5) MergeONNXModels...
Merge terminado en 0.02300548553466797 s
6) Ajustando datatype entrada a UINT8...
7) Limpieza básica...
   InferShapes OK
   FoldConstants OK
   GiveUniqueNodeNames OK
   GiveReadableTensorNames OK
   InferDataTypes OK
   RemoveStaticGraphInputs OK
   RemoveUnusedTensors OK
8) Guardando modelo UINT8...
9) Copiando como model.onnx...
Modelo UINT8 para FINN guardado en: model_for_finn_w4a8_finetuned_uint8.onnx
Copiado como model.onnx
Input: global_in [1, 576] UINT8


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:124: UserWarning: Assuming 2D input is NC
  warnings.warn("Assuming 2D input is NC")


## 15. Evaluar model.onnx con entrada UINT8

In [20]:
# Si la celda anterior creó el preprocesado, model.onnx espera entrada 0-255.
# Si no, puede esperar entrada 0-1. Mira el datatype impreso en la celda anterior.

model = ModelWrapper("model.onnx")
model = model.transform(InferShapes())
input_name = model.graph.input[0].name
output_name = model.graph.output[0].name
print("Input:", input_name, model.get_tensor_shape(input_name), model.get_tensor_datatype(input_name))
print("Output:", output_name, model.get_tensor_shape(output_name))

err_v, err_w = [], []
for i, r in enumerate(rows_eval):
    # probamos con uint8 0-255 porque model.onnx debería incluir /255
    x = image_to_flat_uint8_np(r["img_path"])
    y_dict = execute_onnx(model, {input_name: x})
    y = y_dict[output_name].reshape(-1)
    v_pred = denorm_v_01(float(y[0]))
    w_pred = denorm_w_01(float(y[1]))
    err_v.append(v_pred - r["v"])
    err_w.append(w_pred - r["w"])
    if i < 500:
        print(f"{r['image_name']} | CSV v={r['v']:.4f} w={r['w']:.4f} | model.onnx v={v_pred:.4f} w={w_pred:.4f}")

err_v = np.array(err_v)
err_w = np.array(err_w)
print("MAE_v =", np.mean(np.abs(err_v)))
print("MAE_w =", np.mean(np.abs(err_w)))


Input: global_in [1, 576] UINT8
Output: global_out [1, 2]
1.png | CSV v=5.0000 w=-0.2254 | model.onnx v=4.9801 w=-0.2320
2.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
3.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
4.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
5.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
6.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
7.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
8.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
9.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
10.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
11.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9883 w=-0.2349
12.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
13.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w=-0.2320
14.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9883 w=-0.2349
15.png | CSV v=5.0000 w=-0.2131 | model.onnx v=4.9801 w

131.png | CSV v=5.0000 w=0.1200 | model.onnx v=5.0454 w=0.0656
132.png | CSV v=5.0000 w=0.1200 | model.onnx v=5.0454 w=0.0656
133.png | CSV v=5.0000 w=0.1200 | model.onnx v=5.0454 w=0.0656
134.png | CSV v=5.0000 w=0.1200 | model.onnx v=5.0454 w=0.0741
135.png | CSV v=5.0000 w=0.1200 | model.onnx v=5.0454 w=0.0741
136.png | CSV v=5.0000 w=0.1200 | model.onnx v=5.0474 w=0.0666
137.png | CSV v=5.0000 w=0.1200 | model.onnx v=5.0474 w=0.0666
138.png | CSV v=5.0000 w=0.1147 | model.onnx v=5.0454 w=0.0656
139.png | CSV v=5.0000 w=0.1150 | model.onnx v=5.0454 w=0.0656
140.png | CSV v=5.0000 w=0.1150 | model.onnx v=5.0474 w=0.0666
141.png | CSV v=5.0000 w=0.1150 | model.onnx v=5.0474 w=0.0666
142.png | CSV v=5.0000 w=0.1150 | model.onnx v=5.0474 w=0.0666
143.png | CSV v=5.0000 w=0.1150 | model.onnx v=5.0474 w=0.0666
144.png | CSV v=5.0000 w=0.1150 | model.onnx v=5.0454 w=0.0656
145.png | CSV v=5.0000 w=0.1150 | model.onnx v=5.0454 w=0.0656
146.png | CSV v=5.0000 w=0.1150 | model.onnx v=5.0474 w

262.png | CSV v=5.0000 w=0.0850 | model.onnx v=5.0474 w=0.0468
263.png | CSV v=5.0000 w=0.0850 | model.onnx v=5.0535 w=0.0553
264.png | CSV v=5.0000 w=0.0850 | model.onnx v=5.0535 w=0.0553
265.png | CSV v=5.0000 w=0.0850 | model.onnx v=5.0535 w=0.0553
266.png | CSV v=5.0000 w=0.0850 | model.onnx v=5.0433 w=0.0430
267.png | CSV v=5.0000 w=0.0850 | model.onnx v=5.0433 w=0.0430
268.png | CSV v=5.0000 w=0.0850 | model.onnx v=5.0392 w=0.0402
269.png | CSV v=5.0000 w=0.0850 | model.onnx v=5.0392 w=0.0402
270.png | CSV v=5.0000 w=0.0797 | model.onnx v=5.0392 w=0.0402
271.png | CSV v=5.0000 w=0.0800 | model.onnx v=5.0392 w=0.0402
272.png | CSV v=5.0000 w=0.0800 | model.onnx v=5.0392 w=0.0439
273.png | CSV v=5.0000 w=0.0800 | model.onnx v=5.0392 w=0.0439
274.png | CSV v=5.0000 w=0.0800 | model.onnx v=5.0392 w=0.0402
275.png | CSV v=5.0000 w=0.0800 | model.onnx v=5.0392 w=0.0402
276.png | CSV v=5.0000 w=0.0800 | model.onnx v=5.0392 w=0.0402
277.png | CSV v=5.0000 w=0.0800 | model.onnx v=5.0392 w

393.png | CSV v=5.0000 w=0.0150 | model.onnx v=5.0392 w=0.0063
394.png | CSV v=5.0000 w=0.0097 | model.onnx v=5.0372 w=0.0044
395.png | CSV v=5.0000 w=0.0100 | model.onnx v=5.0372 w=0.0044
396.png | CSV v=5.0000 w=0.0100 | model.onnx v=5.0372 w=0.0044
397.png | CSV v=5.0000 w=0.0100 | model.onnx v=5.0392 w=0.0025
398.png | CSV v=5.0000 w=0.0100 | model.onnx v=5.0331 w=0.0006
399.png | CSV v=5.0000 w=0.0100 | model.onnx v=5.0331 w=0.0006
400.png | CSV v=5.0000 w=0.0100 | model.onnx v=5.0372 w=0.0034
401.png | CSV v=5.0000 w=0.0100 | model.onnx v=5.0372 w=0.0034
402.png | CSV v=5.0000 w=0.0047 | model.onnx v=5.0331 w=0.0006
403.png | CSV v=5.0000 w=0.0050 | model.onnx v=5.0331 w=0.0006
404.png | CSV v=5.0000 w=0.0050 | model.onnx v=5.0331 w=0.0006
405.png | CSV v=5.0000 w=0.0050 | model.onnx v=5.0331 w=0.0006
406.png | CSV v=5.0000 w=0.0050 | model.onnx v=5.0291 w=-0.0079
407.png | CSV v=5.0000 w=0.0050 | model.onnx v=5.0291 w=-0.0079
408.png | CSV v=5.0000 w=0.0050 | model.onnx v=5.0291

## 16. FINN_BUILD_DIR

In [22]:
os.environ["FINN_BUILD_DIR"] = "/tmp/FINN_build_mlp27k_w4a8_ft"
Path(os.environ["FINN_BUILD_DIR"]).mkdir(parents=True, exist_ok=True)
print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])


FINN_BUILD_DIR = /tmp/FINN_build_mlp27k_w4a8_ft


## 17. Intentar build_dataflow

In [1]:
from finn.builder.build_dataflow import build_dataflow_cfg
from finn.builder.build_dataflow_config import (
    DataflowBuildConfig,
    DataflowOutputType,
    ShellFlowType,
    VerificationStepType,
    MemMode,
)
from pathlib import Path
import os


os.environ["FINN_BUILD_DIR"] = "/tmp/FINN_build_mlp27k_w4a8_ft"
Path(os.environ["FINN_BUILD_DIR"]).mkdir(parents=True, exist_ok=True)

MODEL_TO_BUILD = "model.onnx"
OUTPUT_DIR = "output_final_mlp27k_w4a8_ft"

cfg = DataflowBuildConfig(
    output_dir=OUTPUT_DIR,
    synth_clk_period_ns=10.0,
    board=None,
    shell_flow_type=ShellFlowType.NONE,
    generate_outputs=[
        DataflowOutputType.ESTIMATE_REPORTS,
        DataflowOutputType.STITCHED_IP,
        DataflowOutputType.RTLSIM_PERFORMANCE,
        DataflowOutputType.OOC_SYNTH,
    ],
    default_mem_mode=MemMode.INTERNAL_DECOUPLED,
    folding_config_file=None,
    target_fps=100,
    mvau_wwidth_max=36,
    verify_steps=[],
    save_intermediate_models=True,
)

print("Lanzando build_dataflow con", MODEL_TO_BUILD)
print("Output dir:", OUTPUT_DIR)

build_dataflow_cfg(MODEL_TO_BUILD, cfg)

ImportError: cannot import name 'MemMode' from 'finn.builder.build_dataflow_config' (/home/miguel_tfg/FINN-v0.9-Stable/src/finn/builder/build_dataflow_config.py)

## 18. Debug si falla FINN

In [ ]:
!tail -120 output_final_mlp27k_w4a8_ft/build_dataflow.log 2>/dev/null || true


In [ ]:
onnx_files = sorted(Path("/tmp/FINN_build_mlp27k_w4a8_ft").rglob("*.onnx"))
print("ONNX intermedios:", len(onnx_files))
for i, f in enumerate(onnx_files):
    print(i, f)

if onnx_files:
    last_model = onnx_files[-1]
    print("\nInspeccionando:", last_model)
    m = ModelWrapper(str(last_model))
    for node in m.graph.node:
        print(node.name, node.op_type)
